In [7]:
import pandas as pd
from src.load_data import load_parquet
from src.preprocessing import clean_text

df1 = load_parquet("../data_raw/mental_health_vi/train.parquet")
df2 = load_parquet("../data_raw/vi-HealthQA/train.parquet")
df3 = load_parquet("../data_raw/ViHealthQA-small/train.parquet")




In [8]:
df1.head()

,Context_translated,Response_translated
0,Tôi đã trải qua một số điều với cảm xúc và bản...,Nếu tất cả mọi người nghĩ rằng bạn là vô giá t...
1,Tôi đã trải qua một số điều với cảm xúc và bản...,Nếu bạn cảm thấy mình có thể làm được điều gì ...
2,Tôi đã trải qua một số điều với cảm xúc và bản...,Điều đầu tiên tôi đề nghị là có được giấc ngủ ...
3,Tôi đã trải qua một số điều với cảm xúc và bản...,Khi tôi làm việc với những người có cảm giác b...
4,Tôi đã trải qua một số điều với cảm xúc và bản...,Tôi muốn cho bạn biết rằng bạn không phải là n...


In [9]:
df2.head()

,query,positive,negative,cluster
0,Đang chích ngừa viêm gan B có chích ngừa Covid...,[\nNếu anh/chị đang tiêm ngừa vaccine phòng bệ...,[Rất tiếc ở trường hợp của anh/chị không thể t...,134
1,"Đau đầu, căng thẳng do công việc, suy giảm trí...",[\n Nguyên nhân dẫn đến tỷ lệ dịch chuyển tinh...,[Với trường hợp nhiễm nấm đường tiểu ở trẻ nhỏ...,101
2,Bé 13 tháng tuổi uống thuốc Acyclovir có được ...,[\nXin khuyến cáo bạn đưa người nhà đi khám ng...,[Theo quyết định 2995- Bộ Y tế về khám sàng lọ...,157
3,Vừa qua ngày 4/6 tôi có bị con chó ở nhà cắn x...,"[\nTrường hợp của anh/chị, nhịp tim chậm, cần ...",[Bạn đã điều trị lao phổi và đã dừng điều trị ...,243
4,Co giật chi dưới phải điều trị thế nào?,"[\nCó những thức ăn, nước uống mà đôi khi còn ...",[Vùng nách ngoài hạch bạch huyết thường xuất h...,124


In [10]:
df3.head()

,assistant,user
0,Răng bạn hiện tại có mủ dưới lợi gây đau nhức ...,Răng cháu hiện tại có mủ ở dưới lợi nhưng khi ...
1,"Triệu chứng nốt mụn đỏ vùng dưới lưỡi, đau khi...","Em thấy mặt dưới, phía cuống lưỡi của mình có ..."
2,"Các triệu chứng sốt, đau họng, amidan có mủ tá...","Từ tháng 4/2020, em mới xuất hiện lần đầu triệ..."
3,"Các dấu hiệu: nghẹt mũi, vướng họng có thể gặp...","Dạo gần đây, mỗi tối ngủ con hay bị nghẹt mũi ..."
4,Tăng tiết mồ hôi nguyên phát thường do rối loạ...,"Em bị tăng tiết mồ hôi, nhất là vùng mặt và ng..."


In [12]:
import pandas as pd

# 1. Chuẩn hóa cấu trúc các DataFrame trước khi gộp
# df1: Context_translated, Response_translated
df1_p = df1[['Context_translated', 'Response_translated']].rename(
    columns={'Context_translated': 'question', 'Response_translated': 'answer'}
)

# df2: query, positive (lấy phần tử đầu tiên của list)
df2_p = df2[['query', 'positive']].copy()
df2_p['answer'] = df2_p['positive'].apply(lambda x: x[0] if isinstance(x, list) else x)
df2_p = df2_p[['query', 'answer']].rename(columns={'query': 'question'})

# df3: user, assistant
df3_p = df3[['user', 'assistant']].rename(
    columns={'user': 'question', 'assistant': 'answer'}
)

# 2. Gộp tất cả thành một DataFrame duy nhất
df_combined = pd.concat([df1_p, df2_p, df3_p], ignore_index=True)

# 3. Áp dụng làm sạch dữ liệu (sử dụng hàm clean_text bạn đã định nghĩa)
df_combined['question'] = df_combined['question'].apply(clean_text)
df_combined['answer'] = df_combined['answer'].apply(clean_text)

# 4. Lưu thành file CSV
df_combined.to_csv("../data_clean/medical_data_combined.csv", index=False, encoding='utf-8-sig')

# 5. Lưu thành file JSON (định dạng list các object)
df_combined.to_json("../data_clean/medical_data_combined.json", orient='records', force_ascii=False, indent=4)

print("Đã lưu thành công 2 file: medical_data_combined.csv và medical_data_combined.json")

Đã lưu thành công 2 file: medical_data_combined.csv và medical_data_combined.json
